# Ejercicio final de la semana 1

Para demostrar que estás familiarizado con la API de OpenAI y también con Ollama, crea una herramienta que responda a una pregunta técnica
y la explique. ¡Esta es una herramienta que podrás usar durante el curso!

In [1]:
# imports

import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI


In [2]:
# constantes

OLLAMA_API = "http://localhost:11434/api/chat"
HEADERS = {"Content-Type": "application/json"}
MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'


In [ ]:
# Configura el entorno

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key[:8] == 'sk-proj-':
  print("La clave de API parece buena")
else:
  print(
      "¿Puede haber un problema con tu clave API? ¡Visita el cuaderno de resolución de problemas!"
  )

openai = OpenAI()

La clave de API parece buena


In [8]:
# Aquí está la pregunta; escribe sobre esto para preguntar algo nuevo

system_prompt = "Necesito que actues como un programador experto en Python. \
Responde lo más detallado posible, dando ejemplos. \
Responde en formato markdown.\n"

user_prompt = """
Explica qué hace este código y por qué:
yield from {book.get("author") for book in books if book.get("author")}
"""

## GPT

In [ ]:
# Haz que gpt-4o-mini responda, con streaming


def python_expert():
  # Llamada al modelo con streaming habilitado
  stream = openai.chat.completions.create(
      model=MODEL_GPT,
      messages=[
          {
              "role": "system",
              "content": system_prompt
          }, {
              "role": "user",
              "content": user_prompt
          }
      ],
      stream=True  # Habilitar streaming
  )

  # Inicializar variables para mostrar el contenido progresivamente
  response = ""
  display_handle = display(Markdown(""), display_id=True)

  # Procesar cada fragmento de la respuesta
  for chunk in stream:
    response += chunk.choices[0].delta.content or ''
    response = response.replace("```", "").replace("markdown", "")
    update_display(Markdown(response), display_id=display_handle.display_id)

  return response


# Llamada a la función
final_response = python_expert()
print("Respuesta final completa:")
print(final_response)


Claro, puedo explicarte en detalle qué hace ese código. Vamos a descomponerlo paso a paso.

### Descomposición del Código

El código que proporcionaste es:

python
yield from {book.get("author") for book in books if book.get("author")}


1. **`{...}` - Conjunto (Set Comprehension)**:
   - Dentro de las llaves `{}`, estamos utilizando una **comprensión de conjuntos**. Esto significa que estamos creando un conjunto que contiene elementos únicos.

2. **`for book in books`**:
   - Este es un bucle que itera sobre cada elemento en la lista `books`. Supongamos que `books` es una lista de diccionarios, donde cada diccionario representa un libro (por ejemplo, `{"title": "1984", "author": "George Orwell"}`).

3. **`book.get("author")`**:
   - Para cada `book`, utilizamos el método `.get("author")` en el diccionario. Este método intenta obtener el valor asociado a la clave `"author"`. Si la clave no existe, devuelve `None` en lugar de lanzar un error.

4. **`if book.get("author")`**:
   - Este es un **filtro** que asegura que solo se incluya el autor en el conjunto si efectivamente existe. Si `book.get("author")` devuelve `None` o una cadena vacía, entonces no se incluirá ese valor en el conjunto.

5. **`yield from`**:
   - La palabra clave `yield` en Python se usa para generar un valor y hacer que una función se convierta en un generador, lo cual permite retornar varios valores a lo largo del tiempo en lugar de devolver solo uno. La cláusula `from` permite que se itere sobre un iterable, en este caso, el conjunto creado por la comprensión de conjuntos.
   - Entonces, `yield from` devuelve cada elemento del conjunto uno por uno.

### Resumen

En resumen, este código hace lo siguiente:

- Crea un conjunto de nombres de autores únicos a partir de una lista de libros.
- Ignora libros que no tienen un autor especificado (es decir, donde `book.get("author")` es `None` o una cadena vacía).
- Finalmente, utiliza `yield from` para devolver estos autores de manera iterativa.

### Ejemplo Práctico

Imagina que tenemos la siguiente lista de libros:

python
books = [
    {"title": "1984", "author": "George Orwell"},
    {"title": "Brave New World", "author": "Aldous Huxley"},
    {"title": "Fahrenheit 451"},  # Sin autor
    {"title": "The Catcher in the Rye", "author": "J.D. Salinger"},
    {"title": "The Great Gatsby", "author": "F. Scott Fitzgerald"},
    {"title": "Moby Dick", "author": None}  # Sin autor
]


Si utilizaras el código proporcionado en un generador, como en el siguiente ejemplo:

python
def get_authors(books):
    yield from {book.get("author") for book in books if book.get("author")}

# Usando el generador
for author in get_authors(books):
    print(author)


### Salida Esperada

La salida de este código sería:


George Orwell
Aldous Huxley
J.D. Salinger
F. Scott Fitzgerald


### Consideraciones

- El uso de un conjunto asegura que los autores no se repitan, ya que un conjunto en Python no permite duplicados.
- Si se desea mantener el orden de aparición, se podría usar `collections.OrderedDict` o la sintaxis de `dict` desde Python 3.7, donde se garantiza el orden de inserción.

Espero que esta explicación te haya ayudado a entender el funcionamiento del código. Si tienes más preguntas o necesitas asistencia adicional, ¡no dudes en preguntar!

## LLama

### Sin streaming

In [ ]:
# Haz que Llama 3.2 responda

import ollama

messages = [
    {
        "role": "system",
        "content": system_prompt
    }, {
        "role": "user",
        "content": user_prompt
    }
]

payload = {
    "model": MODEL_LLAMA,
    "messages": messages,
    "stream": False,
}

response = ollama.chat(**payload)


In [27]:
import pandas as pd

pd.Series(vars(response))


model                                                            llama3.2
created_at                                 2026-02-07T13:26:23.795616258Z
done                                                                 True
done_reason                                                          stop
total_duration                                                 7252844682
load_duration                                                    61783929
prompt_eval_count                                                      89
prompt_eval_duration                                            203890663
eval_count                                                            688
eval_duration                                                  6605414049
message                 role='assistant' content='**Análisis del códig...
logprobs                                                             None
dtype: object

In [33]:
message_content = response.message.content

# Mostrar en formato Markdown
display(Markdown(message_content[:1000]))

**Análisis del código**
=========================

El código proporcionado es un ejemplo de cómo se puede utilizar la función `yield` en Python para generar un iterador que itera sobre los valores de una lista o colección.

**Estructura del código**
-------------------------

```python
yield from {book.get("author") for book in books if book.get("author")}
```

En este código, podemos identificar las siguientes partes:

*   `yield from`: Esta es la palabra clave que indica el uso de una función llamada "generadora" o "iterador". La función `yield` se utiliza para generar valores en lugar de devolverlos directamente.
*   `{book.get("author") for book in books if book.get("author")}`: Este es un ejemplo de un generador, que es un tipo de iterador que puede ser utilizado con bucles `for`. En este caso, está utilizando el operador `.get()` para acceder a los valores dentro de la lista `books`.

**Funcionamiento del código**
-----------------------------

Aquí hay una explicación paso a pas

### Con streaming

In [35]:
import ollama

messages = [
    {
        "role": "system",
        "content": system_prompt
    },
    {
        "role": "user",
        "content": user_prompt
    },
]

stream = ollama.chat(
    model=MODEL_LLAMA,
    messages=messages,
    stream=True,
)

# Mostrar en vivo en la celda
reply = []
for chunk in stream:
  if "message" in chunk and "content" in chunk["message"]:
    text = chunk["message"]["content"]
    reply.append(text)
    print(text, end="", flush=True)

# Si luego quieres el texto completo:
full_response = "".join(reply)


**Análisis del Código**

El código que se muestra es una expresión genérica en Python, específicamente utilizando la sintaxis de generadores (`yield`). Esta sintaxis permite ejecutar un código y generar valores uno por uno, sin necesidad de crear una lista o estructura de datos completa.

**Explicación Detallada**

El código se puede dividir en dos partes principales:

1. `{book.get("author") for book in books if book.get("author")}`
2. `yield from ...`

**Parte 1: Expresión Genérica**

La primera parte es una expresión genérica que utiliza un iterador (`for`) para iterar sobre un iterable (`books`). El término `if` se aplica a cada elemento del iterable, y solo los elementos que cumplen la condición se procesan.

**Cómo funciona esta expresión:**

* Se itera sobre el iterable `books`.
* Para cada libro (`book`) en el iterable:
	+ Se llama al método `get("author")` para obtener el autor del libro.
	+ Si el libro tiene un autor (es decir, si no es `None` o alguna otra falsa valoración),